## 1. 環境設定

In [ ]:
import os, gc, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
from sentence_transformers import SentenceTransformer

COMP_DIR     = Path('/kaggle/input/competitions/map-charting-student-math-misunderstandings')  
ARTIFACT_DIR = Path('/kaggle/input/datasets/elainecccccc/map-artifacts-20260518-074112/map_artifacts_20260518_074112')  
BGE_DIR      = Path('/kaggle/input/datasets/ethannnchiu/result5/map_artifacts') 

OUT_DIR  = Path('/kaggle/working')
SUB_PATH = OUT_DIR / 'submission.csv'

MAX_LEN    = 256
BATCH_SIZE = 32
MAP_K      = 3
device     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')
assert (COMP_DIR / 'test.csv').exists(), f'找不到 test.csv，路徑錯誤：{COMP_DIR}'
assert (ARTIFACT_DIR / 'deberta_label_list.txt').exists(), '找不到 deberta_label_list.txt'

fold_dirs = sorted([p for p in ARTIFACT_DIR.iterdir()
                    if p.is_dir() and p.name.startswith('deberta_fold')])
assert len(fold_dirs) >= 1
print(f'找到 {len(fold_dirs)} 個 fold: {[p.name for p in fold_dirs]}')

## 2. 載入 test

In [ ]:
def build_input_text(row):
    return (
        "Question: " + str(row['QuestionText']).strip() + "\n"
        "Student selected: " + str(row['MC_Answer']).strip() + "\n"
        "Student explanation: " + str(row['StudentExplanation']).strip()
    )

test = pd.read_csv(COMP_DIR / 'test.csv')
test['text'] = test.apply(build_input_text, axis=1)
print(f'Test shape: {test.shape}')
test.head(2)

## 3. 載入 label list

In [ ]:
with open(ARTIFACT_DIR / 'deberta_label_list.txt') as f:
    LABEL_LIST = [l.strip() for l in f if l.strip()]

# 建立輔助映射
label2idx = {l: i for i, l in enumerate(LABEL_LIST)}

# Category 分組
NA_CATS  = {'True_Correct', 'True_Neither', 'False_Correct', 'False_Neither'}
MIS_CATS = {'True_Misconception', 'False_Misconception'}

cats = sorted({l.split(':')[0] for l in LABEL_LIST})
cat2i = {c: i for i, c in enumerate(cats)}

print(f'Label space: {len(LABEL_LIST)} labels')
print(f'Categories: {cats}')
print(f'NA categories: {NA_CATS}')
print(f'Misconception categories: {MIS_CATS}')

## 4. 模型結構

In [ ]:
class DebertaClassifier(nn.Module):
    def __init__(self, backbone, num_labels):
        super().__init__()
        self.backbone   = backbone
        hidden_size     = backbone.config.hidden_size
        self.pooler     = nn.Linear(hidden_size, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.dropout    = nn.Dropout(0.1)
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None,
                token_type_ids=None, labels=None, **kwargs):
        out    = self.backbone(input_ids=input_ids,
                               attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        cls    = out.last_hidden_state[:, 0, :].float()
        pooled = torch.tanh(self.pooler(self.dropout(cls)))
        logits = self.classifier(self.dropout(pooled))
        loss   = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

print('模型結構定義完成 ✓')

## 5. DeBERTa 5-fold 推論

In [ ]:
@torch.no_grad()
def predict_with_ckpt(ckpt_path, texts):
    tok      = AutoTokenizer.from_pretrained(str(ckpt_path))
    config   = AutoConfig.from_pretrained(str(ckpt_path))
    backbone = AutoModel.from_config(config)
    model    = DebertaClassifier(backbone, num_labels=len(LABEL_LIST))

    state_dict = torch.load(str(ckpt_path / 'pytorch_model_custom.pt'),
                            map_location='cpu', weights_only=True)
    model.load_state_dict(state_dict)
    model.eval().to(device)

    all_probs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        enc   = tok(batch, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors='pt')
        enc   = {k: v.to(device) for k, v in enc.items()}
        probs = torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
    del model, tok, backbone
    torch.cuda.empty_cache()
    return np.concatenate(all_probs, axis=0)

fold_probs = []
for d in fold_dirs:
    print(f'Predicting with {d.name}...')
    fold_probs.append(predict_with_ckpt(d, test['text'].tolist()))

flat_probs = np.mean(fold_probs, axis=0)
print(f'flat_probs shape: {flat_probs.shape}')
assert flat_probs.shape[0] == len(test), f'shape 對不上：{flat_probs.shape[0]} vs {len(test)}'

# 摺疊成 Category 機率（把同一 Category 的所有 label 機率加總）
cat_probs = np.zeros((len(test), len(cats)), dtype=np.float32)
for i, l in enumerate(LABEL_LIST):
    cat_probs[:, cat2i[l.split(':')[0]]] += flat_probs[:, i]

print(f'cat_probs shape: {cat_probs.shape}')
print(f'Sample: {dict(zip(cats, cat_probs[0].round(3)))}')

## 6. BGE 檢索器（Global + Local Hybrid Anchor）

### 設計說明

**兩層 anchor 策略：**
1. **Global anchor**：所有 train 裡同 Misconception 的 explanation 取平均向量 → 對所有 test 都有效
2. **Local anchor**：同 QuestionId + 同 Misconception 取平均 → 對已知 QuestionId 更精準

推論時：對每個 test 樣本，如果它的 QuestionId 在 local anchor 裡，用 local；否則用 global。

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

print('Loading fine-tuned BGE model (HuggingFace Native)...')
bge_path = str(BGE_DIR / 'bge_finetuned')

bge_tokenizer = AutoTokenizer.from_pretrained(bge_path)
bge_model = AutoModel.from_pretrained(bge_path).to(device).eval()

print('Encoding test texts...')
test_texts = test['text'].tolist()
all_embeddings = []

with torch.no_grad():
    for i in range(0, len(test_texts), 64):
        batch_texts = test_texts[i : i+64]
        inputs = bge_tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        
        outputs = bge_model(**inputs)
        
        last_hidden_state = outputs.last_hidden_state
        input_mask_expanded = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
        
        # 計算所有有效 Token 的總和與數量
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)  # 防止除以 0
        
        # 取得真正的 Mean Pooling 向量
        embeddings = sum_embeddings / sum_mask
        
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())

test_embeddings = np.vstack(all_embeddings).astype(np.float32)
print(f'test_embeddings shape: {test_embeddings.shape}')


# 載入 local anchor
with open(BGE_DIR / 'local_anchors.pkl', 'rb') as f:
    local_anchors = pickle.load(f)

# 從 local_anchors 建出 global_anchors
global_anchors = {}
for qid, misc_dict in local_anchors.items():
    for misc, emb in misc_dict.items():
        if misc not in global_anchors:
            global_anchors[misc] = []
        global_anchors[misc].append(emb)
global_anchors = {m: np.mean(np.vstack(v), axis=0) for m, v in global_anchors.items()}

# L2 normalize global anchors
for m in global_anchors:
    v = global_anchors[m]
    global_anchors[m] = v / (np.linalg.norm(v) + 1e-10)

print(f'Global anchors: {len(global_anchors)} misconceptions')
print(f'Local anchors: {len(local_anchors)} questions')

# 建立「Misconception 名稱 → label index」映射
mis_to_label_idx = {}
for i, l in enumerate(LABEL_LIST):
    parts = l.split(':')
    if len(parts) == 2:
        mis = parts[1]
        if mis != 'NA':
            if mis not in mis_to_label_idx:
                mis_to_label_idx[mis] = []
            mis_to_label_idx[mis].append(i)

print(f'Misconception types in label space: {len(mis_to_label_idx)}')

## 7. 真正的兩階段融合

### 邏輯說明

```
對每筆 test 樣本：

1. DeBERTa 的 Category 機率 → 判斷最可能的 category group
   - NA group (*_Correct, *_Neither): 強制 Misconception = NA
   - Misconception group (*_Misconception): 進入第二階段

2. 第二階段（僅 Misconception 類）：
   - 用 BGE 找最相似的 Misconception anchor
   - 融合分數 = category_prob × retrieval_sim
   - 取 top-3
```

In [ ]:
def two_stage_predict(
    flat_probs,        # (N, L) DeBERTa 機率
    cat_probs,         # (N, 6) Category 機率
    test_embeddings,   # (N, D) BGE embedding
    test_df,
    local_anchors,
    global_anchors,
    mis_to_label_idx,
    LABEL_LIST,
    cats,
    cat2i,
    NA_CATS,
    MIS_CATS,
    deberta_weight=0.85,  
    k=3,
):
    N = len(flat_probs)
    all_top_labels = []

    for n in range(N):
        cp = cat_probs[n]
        
        na_scores = {}
        for c in NA_CATS:
            if c in cat2i:
                na_scores[f'{c}:NA'] = cp[cat2i[c]]

        mis_scores = {}
        mis_cat_total_prob = sum(cp[cat2i[c]] for c in MIS_CATS if c in cat2i)
        
        # 找出 DeBERTa 在這個樣本的最高信心度
        max_deberta_prob = np.max(flat_probs[n])

        if mis_cat_total_prob > 0.01 and max_deberta_prob < 0.90:
            q_emb = test_embeddings[n]
            qid = test_df.iloc[n]['QuestionId']

            def get_anchor(mis_name):
                if qid in local_anchors and mis_name in local_anchors[qid]:
                    v = local_anchors[qid][mis_name]
                    return v / (np.linalg.norm(v) + 1e-10)
                return None

            for mis_name, label_indices in mis_to_label_idx.items():
                anchor = get_anchor(mis_name)
                if anchor is None:
                    continue
                    
                sim = float(q_emb @ anchor)
                sim = max(sim, 0)
                sim = sim ** 2 

                for c in MIS_CATS:
                    if c not in cat2i:
                        continue
                    label_key = f'{c}:{mis_name}'
                    if label_key not in {l: i for i, l in enumerate(LABEL_LIST)}:
                        continue
                        
                    cat_p = cp[cat2i[c]]
                    label_idx_list = [i for i, l in enumerate(LABEL_LIST) if l == label_key]
                    
                    if label_idx_list:
                        deberta_p = flat_probs[n, label_idx_list[0]]
                    else:
                        deberta_p = cat_p * 0.1
                        
                    # 融合分數
                    fused = deberta_weight * deberta_p + (1 - deberta_weight) * (cat_p * sim)
                    mis_scores[label_key] = fused

        if not mis_scores:
            for c in MIS_CATS:
                if c in cat2i:
                    for mis_name, label_indices in mis_to_label_idx.items():
                        label_key = f'{c}:{mis_name}'
                        label_idx_list = [i for i, l in enumerate(LABEL_LIST) if l == label_key]
                        if label_idx_list:
                            mis_scores[label_key] = flat_probs[n, label_idx_list[0]]

        all_scores = {**na_scores, **mis_scores}
        sorted_labels = sorted(all_scores, key=lambda x: -all_scores[x])
        top_k = sorted_labels[:k]

        fallbacks = ['False_Neither:NA', 'True_Correct:NA', 'False_Misconception:Incomplete']
        for fb in fallbacks:
            if len(top_k) >= k:
                break
            if fb not in top_k:
                top_k.append(fb)

        all_top_labels.append(top_k)

    return all_top_labels

In [ ]:
print('Running two-stage prediction...')
top_labels = two_stage_predict(
    flat_probs      = flat_probs,
    cat_probs       = cat_probs,
    test_embeddings = test_embeddings,
    test_df         = test,
    local_anchors   = local_anchors,
    global_anchors  = global_anchors,
    mis_to_label_idx = mis_to_label_idx,
    LABEL_LIST      = LABEL_LIST,
    cats            = cats,
    cat2i           = cat2i,
    NA_CATS         = NA_CATS,
    MIS_CATS        = MIS_CATS,
    deberta_weight  = 0.85, 
)

print(f'Predictions done: {len(top_labels)} rows')
print('Sample predictions:')
for i in range(min(5, len(top_labels))):
    print(f'  [{i}] {top_labels[i]}')

## 8. 輸出 submission.csv

In [ ]:
sub = pd.DataFrame({
    'row_id': test['row_id'].values,
    'Category:Misconception': [' '.join(ls) for ls in top_labels],
})
sub.to_csv(SUB_PATH, index=False)
print(f'Saved: {SUB_PATH}  ({len(sub)} rows)')

# 格式驗證
check = pd.read_csv(SUB_PATH)
assert list(check.columns) == ['row_id', 'Category:Misconception']
assert len(check) == len(test), f'列數錯：{len(check)} vs {len(test)}'

valid_cats = {'True_Correct','True_Neither','True_Misconception',
              'False_Correct','False_Neither','False_Misconception'}
for i, row in check.head(100).iterrows():
    preds = row['Category:Misconception'].split(' ')
    assert 1 <= len(preds) <= 3
    for p in preds:
        assert ':' in p and p.split(':')[0] in valid_cats, f'格式錯：{p}'

print('格式檢查通過 ✓')
check.head(5)